In [1]:
import numpy as np 
import cbor2

In [2]:
import cbor2

from pathlib import Path
base_dir = Path("benchmarkY2test-public.v2.1.1/benchmarkY2.public")
file_path = base_dir / "benchmarkY2.cbor-outlines.cbor"

data= []

with open(file_path, "rb") as f:
    for i in range(2): # only 2 objects !
        obj = cbor2.load(f)
        data.append(obj)
        print(type(obj))
        print(obj)
        print("=" * 50)

<class 'list'>
['CAR', [1], [0, [[0, 'tqa', 'en-us', 'tqa', []], [0, 'enwiki', 'en-US', '20180601', []]], 'benchmarkY2test', [], []]]
<class 'list'>
[[1, 'relative ages of rocks', b'tqa:relative%20ages%20of%20rocks', [[0, 'Matching Rock Layers', b'Matching%20Rock%20Layers', []], [0, 'Putting Events in Order', b'Putting%20Events%20in%20Order', []], [0, 'Law of Superposition', b'Law%20of%20Superposition', []], [0, 'Widespread Rock Layers', b'Widespread%20Rock%20Layers', []], [0, 'Law of Lateral Continuity', b'Law%20of%20Lateral%20Continuity', []], [0, 'Dividing Geologic Time', b'Dividing%20Geologic%20Time', []], [0, 'The Geologic Time Scale', b'The%20Geologic%20Time%20Scale', []], [0, 'Law of Original Horizontality', b'Law%20of%20Original%20Horizontality', []], [0, 'Your Place in Geologic Time', b'Your%20Place%20in%20Geologic%20Time', []], [0, 'Law of CrossCutting Relationships', b'Law%20of%20CrossCutting%20Relationships', []], [0, 'Using Index Fossils', b'Using%20Index%20Fossils', []], 

In [3]:
# forma first element
def parse_car_header(obj):
    return {
        "format": obj[0],             
        "version": obj[1][0],         
        "corpora": [
            {
                "id": x[1],
                "lang": x[2],
                "source": x[3]
            }
            for x in obj[2][1]
        ],
        "dataset": obj[2][2]
    }

parse_car_header(data[0])

{'format': 'CAR',
 'version': 1,
 'corpora': [{'id': 'tqa', 'lang': 'en-us', 'source': 'tqa'},
  {'id': 'enwiki', 'lang': 'en-US', 'source': '20180601'}],
 'dataset': 'benchmarkY2test'}

In [4]:
def parse_topic(node):
    """
    Recursively parse CAR topic tree
    """
    level = node[0]
    title = node[1]
    topic_id = node[2].decode("utf-8") if isinstance(node[2], bytes) else node[2]

    children = []

    # node[3] contains subtopics
    for child in node[3]:
        children.append(parse_topic(child))

    return {
        "level": level,
        "title": title,
        "id": topic_id,
        "children": children
    }

def parse_all_topics(raw_topics):
    return [parse_topic(t) for t in raw_topics]

In [5]:
topics_clean = parse_all_topics(data[1])  # second list item
topics_clean

[{'level': 1,
  'title': 'relative ages of rocks',
  'id': 'tqa:relative%20ages%20of%20rocks',
  'children': [{'level': 0,
    'title': 'Matching Rock Layers',
    'id': 'Matching%20Rock%20Layers',
    'children': []},
   {'level': 0,
    'title': 'Putting Events in Order',
    'id': 'Putting%20Events%20in%20Order',
    'children': []},
   {'level': 0,
    'title': 'Law of Superposition',
    'id': 'Law%20of%20Superposition',
    'children': []},
   {'level': 0,
    'title': 'Widespread Rock Layers',
    'id': 'Widespread%20Rock%20Layers',
    'children': []},
   {'level': 0,
    'title': 'Law of Lateral Continuity',
    'id': 'Law%20of%20Lateral%20Continuity',
    'children': []},
   {'level': 0,
    'title': 'Dividing Geologic Time',
    'id': 'Dividing%20Geologic%20Time',
    'children': []},
   {'level': 0,
    'title': 'The Geologic Time Scale',
    'id': 'The%20Geologic%20Time%20Scale',
    'children': []},
   {'level': 0,
    'title': 'Law of Original Horizontality',
    'id': '

In [6]:
topics_clean[0] # ['children']

{'level': 1,
 'title': 'relative ages of rocks',
 'id': 'tqa:relative%20ages%20of%20rocks',
 'children': [{'level': 0,
   'title': 'Matching Rock Layers',
   'id': 'Matching%20Rock%20Layers',
   'children': []},
  {'level': 0,
   'title': 'Putting Events in Order',
   'id': 'Putting%20Events%20in%20Order',
   'children': []},
  {'level': 0,
   'title': 'Law of Superposition',
   'id': 'Law%20of%20Superposition',
   'children': []},
  {'level': 0,
   'title': 'Widespread Rock Layers',
   'id': 'Widespread%20Rock%20Layers',
   'children': []},
  {'level': 0,
   'title': 'Law of Lateral Continuity',
   'id': 'Law%20of%20Lateral%20Continuity',
   'children': []},
  {'level': 0,
   'title': 'Dividing Geologic Time',
   'id': 'Dividing%20Geologic%20Time',
   'children': []},
  {'level': 0,
   'title': 'The Geologic Time Scale',
   'id': 'The%20Geologic%20Time%20Scale',
   'children': []},
  {'level': 0,
   'title': 'Law of Original Horizontality',
   'id': 'Law%20of%20Original%20Horizontalit

In [7]:
len(topics_clean)

65

In [8]:
file_path = base_dir / "benchmarkY2.titles"

titles = [] # titles found in roots 

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        titles.append(line.strip())

In [9]:
len(titles), titles[:5]

(65,
 ['relative ages of rocks',
  'absolute ages of rocks',
  'energy in the atmosphere',
  'world climates',
  'climate change'])

In [10]:
file_path = base_dir / "benchmarkY2.topics"

topics = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        topics.append(line.strip())
topics[:5], len(set(topics))

(['enwiki:Aerobic%20fermentation/Aerobic%20fermentation%20in%20non-yeast%20species',
  'enwiki:Aerobic%20fermentation/Aerobic%20fermentation%20in%20non-yeast%20species/E.%20coli%20mutant',
  'enwiki:Aerobic%20fermentation/Aerobic%20fermentation%20in%20non-yeast%20species/Plants',
  'enwiki:Aerobic%20fermentation/Aerobic%20fermentation%20in%20non-yeast%20species/Trypanosomatids',
  'enwiki:Aerobic%20fermentation/Aerobic%20fermentation%20in%20non-yeast%20species/Tumor%20cells'],
 1021)

In [11]:
total = 65

for el in topics_clean:
    total += len(el["children"])
total

622

Loading qrels

In [12]:
def load_qrels(path):
    qrels = {}

    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            qid, _, docid, rel = line.strip().split()
            qid = int(qid)
            rel = int(rel)

            if qid not in qrels:
                qrels[qid] = {}

            qrels[qid][docid] = rel

    return qrels

def load_entity_qrels(path):
    qrels = {}

    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()

            # handle 3-column format
            if len(parts) == 3:
                qid, entity, rel = parts
                rel = int(rel)

            # handle 4-column format (just in case)
            elif len(parts) == 4:
                qid, _, entity, rel = parts
                rel = int(rel)

            else:
                continue

            qrels.setdefault(qid, {})[entity] = rel

    return qrels

In [14]:
path = "benchmarkY2test-auto-manual-qrels/benchmarkY2test/benchmarkY2test-entity-automatic.qrels"

qrels_entity = load_entity_qrels(path)

In [15]:
len(qrels_entity.keys())

976

In [21]:
qrels_entity['enwiki:Aerobic%20fermentation']

KeyError: 'enwiki:Aerobic%20fermentation'

In [23]:
path = "benchmarkY2test-auto-manual-qrels/benchmarkY2test/benchmarkY2test-entity-manual.qrels"

qrels = load_entity_qrels(path)

In [24]:
len(qrels)

271

In [26]:
qrels.keys()

dict_keys(['enwiki:Aerobic%20fermentation/Aerobic%20fermentation%20in%20non-yeast%20species/E.%20coli%20mutant', 'enwiki:Aerobic%20fermentation/Aerobic%20fermentation%20in%20non-yeast%20species/Plants', 'enwiki:Aerobic%20fermentation/Aerobic%20fermentation%20in%20non-yeast%20species/Trypanosomatids', 'enwiki:Aerobic%20fermentation/Aerobic%20fermentation%20in%20non-yeast%20species/Tumor%20cells', 'enwiki:Aerobic%20fermentation/Domestication%20and%20aerobic%20fermentation', 'enwiki:Aerobic%20fermentation/Genomic%20basis%20of%20the%20crabtree%20effect', 'enwiki:Aerobic%20fermentation/Genomic%20basis%20of%20the%20crabtree%20effect/CNV%20in%20fermentation%20genes', 'enwiki:Aerobic%20fermentation/Genomic%20basis%20of%20the%20crabtree%20effect/CNV%20in%20glycolysis%20genes', 'enwiki:Aerobic%20fermentation/Genomic%20basis%20of%20the%20crabtree%20effect/Differential%20expression', 'enwiki:Aerobic%20fermentation/Genomic%20basis%20of%20the%20crabtree%20effect/Expansion%20of%20hexose%20transporter